# Hands-on Lab — Small Transformers in Practice: One Input, Three Architectures

*Statistical Foundations of LLMs · Session 2 · Accompanies slides 252–260*

Transformers come in three flavors — **encoder-only** (BERT family), **decoder-only** (GPT family), and **encoder–decoder** (T5 family). The best way to understand the difference is to feed **the same sentence to all three** and watch what each one does with it.

All models here are small enough to run on the free Colab tier (CPU works too, just slower).


## Learning Objectives

1. Run inference with **DistilBERT** (classification), **GPT-2** (generation), and **T5-small** (text-to-text).
2. Compare how the three architectures tokenize and process identical input.
3. Match architecture to task: understanding vs. free generation vs. structured transformation.
4. Observe practical trade-offs: speed, output form, prompt sensitivity, hallucination.


## 0. New to Jupyter? Start Here (2 minutes)

**What is a Jupyter Notebook?** A document that mixes text and runnable Python code, organized in *cells*.

| What you need to know | How |
|---|---|
| **Run a cell** | Click it, then press **Shift + Enter** (or the ▶ button) |
| **Cell types** | **Markdown** cells = formatted text (like this one). **Code** cells = Python you can execute |
| **Order matters** | Run cells **top to bottom**. A cell may depend on variables defined above it |
| **Restart the kernel** | Menu: *Runtime → Restart runtime* (Colab) or *Kernel → Restart* (Jupyter). Then re-run cells from the top |
| **Install packages** | Run a cell starting with `%pip install ...`, then restart the kernel if asked |
| **Modify code** | Just edit any code cell and re-run it — experimenting is the whole point! |
| **Read outputs** | Results appear directly below each code cell: printed text, tables, or plots |

> 💡 **Tip:** If something behaves strangely, *Restart runtime* and run all cells from the top (*Runtime → Run all*).


## 1. Background: Three Architectures at a Glance

| | Encoder-only | Decoder-only | Encoder–decoder |
|---|---|---|---|
| **Example** | DistilBERT, ALBERT | GPT-2, GPT-4 | T5, BART |
| **Attention** | Bidirectional (sees whole sentence) | Causal (left-to-right only) | Both |
| **Output** | Label / embedding | Free-form continuation | New sequence |
| **Natural task** | Classification, NER | Storytelling, chat | Summarization, translation |
| **Analogy** | A *reader* | A *storyteller* | A *translator* |


## 2. Setup and Imports


In [ ]:
# Run once if needed (sentencepiece is required by the T5 tokenizer):
# %pip install -U transformers torch pandas sentencepiece --quiet


In [ ]:
import pandas as pd
import torch
from transformers import pipeline, AutoTokenizer, set_seed

set_seed(42)
DEVICE = 0 if torch.cuda.is_available() else -1
print("Using:", torch.cuda.get_device_name(0) if DEVICE == 0 else "CPU")


## 3. Encoder-Only: DistilBERT for Classification

DistilBERT reads the **entire sentence at once** (bidirectional attention) and outputs a **label**, not text.


In [ ]:
classifier = pipeline(
    task="text-classification",
    model="distilbert-base-uncased-finetuned-sst-2-english",
    device=DEVICE,
)

print(classifier("I hate using Hugging Face Transformers!"))
print(classifier("The film was surprisingly good."))


**What we observe:** a label plus a calibrated confidence score — fast and accurate for clear inputs.

**Discussion point:** this is *classification, not generation* — the model cannot produce natural language.

**Key takeaway:** encoder-only models are efficient for structured prediction (sentiment, NER, spam filtering).


## 4. Decoder-Only: GPT-2 for Open-Ended Generation

GPT-2 predicts the next token from left to right — it will happily continue *anything*, but follows no instructions unless the prompt embeds them.


In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel

gpt2_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
gpt2_model = GPT2LMHeadModel.from_pretrained("gpt2")

def gpt2_generate(prompt, max_new_tokens=40, seed=42):
    set_seed(seed)
    inputs = gpt2_tokenizer(prompt, return_tensors="pt")
    outputs = gpt2_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_k=50,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
    return gpt2_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(gpt2_generate("Once upon a time,"))


### ✏️ Exercise 4.1 — Prompt sensitivity

Run the two prompts below. Same *task intention*, very different behavior — why?


In [ ]:
print("A:", gpt2_generate("The movie was great.", seed=1))
print()
print("B:", gpt2_generate("Review: The movie was great. Sentiment (positive/negative):", seed=1))


**What we observe:** GPT-2 produces fluent continuations, sometimes creative or off-topic. Prompt B *nudges* it toward classification, but with no guarantees.

**Key takeaway:** decoder-only models excel at free-form generation but need careful prompting to stay on task — the seed of *prompt engineering* (Session 3).


## 5. Encoder–Decoder: T5 for Task-Directed Transformation

T5 casts **every task as text-to-text**, with the task named in the input prefix (`summarize:`, `translate English to German:`).


In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration

t5_tokenizer = T5Tokenizer.from_pretrained("t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("t5-small")

def t5_run(input_text, max_length=40):
    inputs = t5_tokenizer(input_text, return_tensors="pt")
    outputs = t5_model.generate(**inputs, max_length=max_length)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

print(t5_run("summarize: The movie was too long, but the acting was brilliant and the plot was engaging."))
print(t5_run("translate English to German: The weather is beautiful today."))


**Key takeaway:** T5 combines bidirectional understanding (encoder) with generation (decoder) — one model, many tasks, task selection via the prefix.


## 6. Comparative Exercise — One Input, Three Models

Now the main event (slide 255). Run the **same sentence** through all three and record the outputs.


In [ ]:
SENTENCE = "The movie started slowly but turned out to be incredibly powerful and well-acted."

results = {
    "DistilBERT (encoder-only)": str(classifier(SENTENCE)[0]),
    "GPT-2 (decoder-only)":      gpt2_generate(SENTENCE, max_new_tokens=30),
    "T5-small (encoder-decoder)": t5_run("summarize: " + SENTENCE),
}

for model, output in results.items():
    print(f"### {model}\n{output}\n")


### ✏️ Exercise 6.1 — Your turn

1. Replace `SENTENCE` with a review **you** write (try one with mixed sentiment).
2. Record each model's output in the table below (double-click this cell to edit).

| Model | Output form | Useful for... |
|---|---|---|
| DistilBERT | | |
| GPT-2 | | |
| T5-small | | |

**Reflection (from slide 255):**
- How did the outputs differ in *form* and *style*?
- Which model gave the most useful response for *your* intended purpose?
- What does this tell you about choosing an architecture?


## 7. Bonus Comparison: Same Sentence, Three Tokenizers

The differences start *before* the model: each family tokenizes differently.


In [ ]:
tokenizers = {
    "DistilBERT (WordPiece)": AutoTokenizer.from_pretrained("distilbert-base-uncased"),
    "GPT-2 (BPE)":            gpt2_tokenizer,
    "T5 (SentencePiece)":     t5_tokenizer,
}

rows = []
for name, tok in tokenizers.items():
    ids = tok(SENTENCE)["input_ids"]
    rows.append({
        "tokenizer": name,
        "token_count": len(ids),
        "first_8_tokens": tok.convert_ids_to_tokens(ids)[:8],
    })
pd.DataFrame(rows)


### ⏱️ Speed check

Which model responds fastest? (Slide 257 asks this — now measure it.)


In [ ]:
import time

def time_it(fn, *args, n=3):
    start = time.perf_counter()
    for _ in range(n):
        fn(*args)
    return (time.perf_counter() - start) / n

timings = {
    "DistilBERT": time_it(classifier, SENTENCE),
    "GPT-2":      time_it(gpt2_generate, SENTENCE),
    "T5-small":   time_it(t5_run, "summarize: " + SENTENCE),
}
pd.Series(timings, name="seconds/call").round(2).to_frame()


## 8. 🏆 Challenge Exercises

**Challenge A — Fill-mask.** Encoder models were pretrained on masked prediction. Try `pipeline("fill-mask", model="distilbert-base-uncased")` on `"The film was [MASK] and surprisingly emotional."`. Compare the top-5 candidates with what GPT-2 generates after `"The film was"`.

**Challenge B — Make GPT-2 classify.** Design a few-shot prompt (2–3 labeled examples inside the prompt) that makes GPT-2 output only `positive` or `negative`. Measure its accuracy on 5 sentences vs. DistilBERT. *(Preview of in-context learning, Session 3.)*

**Challenge C — Push T5.** Try `question: ... context: ...` for QA, or ask T5-small to summarize a 300-word paragraph. Where does the small model break down?


In [ ]:
# Challenge workspace:


## 9. Discussion Questions (slides 257–258)

1. When would you choose encoder-only vs. decoder-only vs. encoder–decoder? Give one concrete application each.
2. Which model gave calibrated confidence scores? Which only produced text? Why does that matter for *statistical* use of model outputs?
3. GPT-2 sometimes invents facts in its continuations. Did you observe this? *(Foreshadows the hallucination case study.)*
4. Do smaller models suffice for lightweight applications? What practical limits did you observe today (speed, quality, prompt sensitivity)?


## Key Takeaways

- **Encoder-only** = reader → labels & embeddings; fast, calibrated, cannot generate.
- **Decoder-only** = storyteller → fluent free text; needs prompting, can hallucinate.
- **Encoder–decoder** = translator → structured input→output transformations, task-aware via prefixes.
- Architecture choice is a *statistical modeling decision*: what is your input, what is your output, and do you need probabilities or prose?

## References

- Vaswani et al. (2017), *Attention Is All You Need* — https://arxiv.org/abs/1706.03762
- Raffel et al. (2020), *Exploring the Limits of Transfer Learning with T5* — https://arxiv.org/abs/1910.10683
- Sanh et al. (2019), *DistilBERT* — https://arxiv.org/abs/1910.01108
